# Lab: triagem de mamografias

Notebook de aula da página: https://insper.github.io/ML/pt/2026.2/classes/roc-imbalanced/

Rode as células em ordem. Em cada passo, responda o **Preveja** com a dupla *antes* de rodar. Nas **Simulações**, escreva o experimento na célula vazia; as soluções e a leitura de cada resultado estão na página da aula.

## Passo 1 — carregar e olhar a prevalência

**Preveja.** Microcalcificações são achados raros numa mamografia. Antes de rodar: você espera que elas sejam mais ou menos de 10% das regiões? Com essa prevalência, quanto de acurácia um modelo que nunca sinaliza nada conseguiria?

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_openml

mammo = fetch_openml(data_id=310, as_frame=True)
X = mammo.data                                   # 6 atributos numéricos por região
y = (mammo.target == '1').astype(int)            # o alvo é '-1' (benigno) ou '1' (maligno)
print(X.shape)
print(y.value_counts())
print(f"prevalence: {y.mean():.2%}")

## Passo 2 — divisão estratificada treino / validação / teste

**Preveja.** Por que estratificar importa mais aqui do que num conjunto equilibrado? Se a validação tiver cerca de 50 casos malignos, quanto de recall vale um único caso?

In [ ]:
from sklearn.model_selection import train_test_split

X_tmp, X_test, y_tmp, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=7)
X_train, X_val, y_train, y_val = train_test_split(X_tmp, y_tmp, test_size=0.25,
                                                  stratify=y_tmp, random_state=7)
for name, part in [('train', y_train), ('validation', y_val), ('test', y_test)]:
    print(f"{name:10s} {len(part):5d} rows  {part.sum():3d} malignant  ({part.mean():.2%})")

## Passo 3 — um baseline que nunca sinaliza nada

**Preveja.** Qual vai ser a acurácia de responder "benigno" para todas as regiões? E o recall?

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, recall_score

dummy = DummyClassifier(strategy='most_frequent').fit(X_train, y_train)
pred = dummy.predict(X_val)
print(f"accuracy {accuracy_score(y_val, pred):.4f}   recall {recall_score(y_val, pred):.3f}")

## Passo 4 — uma regressão logística no limiar padrão

**Preveja.** A acurácia vai superar a do baseline por muito ou por pouco? E o recall em 0,5: você espera perto de 0,9 ou bem abaixo disso, dado que só 2,3% das regiões são malignas?

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, precision_score, roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

lr = make_pipeline(StandardScaler(), LogisticRegression()).fit(X_train, y_train)
p_val = lr.predict_proba(X_val)[:, 1]
pred = (p_val >= 0.5).astype(int)
print(f"accuracy {accuracy_score(y_val, pred):.4f}   flagged {pred.sum()} of {len(pred)}")
print(f"recall {recall_score(y_val, pred):.3f}   precision {precision_score(y_val, pred):.3f}")
print(f"ROC AUC {roc_auc_score(y_val, p_val):.3f}   AP {average_precision_score(y_val, p_val):.3f}"
      f"   (AP of a random scorer = {y_val.mean():.3f})")

**Simulação: a AUC só enxerga a ordem.** Pegue os escores `p_val` e transforme-os de maneiras que **preservam a ordem** (elevar ao cubo, virar chances $p/(1-p)$) e de uma que a **inverte** ($1-p$). Para cada versão, meça a AUC, a AP e quantas regiões ficam acima de 0,5. O que muda e o que não muda?

In [ ]:
# sua simulação aqui

## Passo 5 — as curvas ROC e PR

**Preveja.** Para chegar a recall 0,80 neste conjunto, você acha que a precisão fica mais perto de 0,8 ou de 0,3? Lembre que só 2,3% das regiões são malignas.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import PrecisionRecallDisplay, RocCurveDisplay, precision_recall_curve, roc_curve

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
RocCurveDisplay.from_predictions(y_val, p_val, ax=ax1, plot_chance_level=True)
PrecisionRecallDisplay.from_predictions(y_val, p_val, ax=ax2, plot_chance_level=True)
plt.show()

fpr, tpr, _ = roc_curve(y_val, p_val)
for max_fpr in [0.01, 0.05, 0.10]:
    print(f"FPR <= {max_fpr:.2f}: best recall {tpr[fpr <= max_fpr].max():.3f}")
prec, rec, _ = precision_recall_curve(y_val, p_val)
for min_rec in [0.5, 0.8, 0.9]:
    print(f"recall >= {min_rec:.1f}: best precision {prec[rec >= min_rec].max():.3f}")

**Simulação: o preço de cada ponto de recall.** Para metas de recall de 0,50 a 0,95, encontre na validação o limiar que atinge a meta com a melhor precisão e conte quantas regiões ele sinaliza. Traduza a precisão em algo que um radiologista entenda: quantas regiões benignas ele revisa para cada câncer encontrado?

In [ ]:
# sua simulação aqui

**Para fechar a parte fundamental.** Em uma frase cada: (1) por que a curva ROC não muda quando a doença fica mais rara e a curva PR muda? (2) O que a AUC mede que a acurácia em 0,5 não mede? (3) Quem decide o ponto de operação, o treino ou você?

## Sua vez (fundamental)

Para casa:

1. **A escada à mão.** Pegue os 10 primeiros pacientes do exemplo das 20 linhas e desenhe a curva ROC deles no papel, um degrau por paciente. Confira com `roc_curve`. Em que ponto está o limiar 0,5?
2. **AUC sem calibração.** Comprima os escores da validação do passo 4 para a faixa de 0,40 a 0,41 com `0.40 + 0.01 * p_val`. O que acontece com a AUC e o que acontece com a acurácia em 0,5? Que métrica você usaria para descobrir que as probabilidades estão erradas?
3. **AP contra a linha de base.** Calcule a AP de um escorador aleatório (use `rng.random(len(y_val))`) e compare com a AP do modelo e com a prevalência. Quantas vezes melhor que o acaso o modelo é?

## Passo 6 — pesos de classe

**Preveja.** Com `class_weight='balanced'`, o que você espera que mude mais: a ROC AUC (a ordenação) ou o número de regiões sinalizadas em 0,5?

In [ ]:
lr_w = make_pipeline(StandardScaler(), LogisticRegression(class_weight='balanced')).fit(X_train, y_train)
p_val_w = lr_w.predict_proba(X_val)[:, 1]
pred = (p_val_w >= 0.5).astype(int)
print(f"flagged {pred.sum()}   recall {recall_score(y_val, pred):.3f}   precision {precision_score(y_val, pred):.3f}")
print(f"ROC AUC {roc_auc_score(y_val, p_val_w):.3f}   AP {average_precision_score(y_val, p_val_w):.3f}")
n0, n1 = np.bincount(y_train)
print(f"intercept shift {lr_w[-1].intercept_[0] - lr[-1].intercept_[0]:.2f}   log(n0/n1) = {np.log(n0 / n1):.2f}")

**Simulação: o limiar que imita os pesos.** Se os pesos basicamente deslocam o ponto de operação, dá para chegar perto sem retreinar: corte os escores do modelo **simples** na prevalência do treino em vez de em 0,5. Compare esse ponto com o do modelo com pesos em 0,5 e conte em quantas regiões as duas regras discordam.

In [ ]:
# sua simulação aqui

## Passo 7 — reamostragem dentro de um pipeline do `imblearn`

**Preveja.** A subamostragem vai treinar em quantas linhas, mais ou menos? E o SMOTE? Qual das duas joga informação fora?

In [ ]:
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import make_pipeline as make_imb_pipeline
from imblearn.under_sampling import RandomUnderSampler

models = {
    'logistic': lr,
    'logistic, balanced': lr_w,
    'undersampling': make_imb_pipeline(StandardScaler(), RandomUnderSampler(random_state=0),
                                       LogisticRegression()),
    'SMOTE': make_imb_pipeline(StandardScaler(), SMOTE(random_state=0), LogisticRegression()),
}
for name in ['undersampling', 'SMOTE']:
    models[name].fit(X_train, y_train)            # o amostrador roda aqui, só no X_train
    X_r, y_r = models[name][1].fit_resample(X_train, y_train)
    print(f"{name:14s} trains on {len(y_r):5d} rows, {y_r.sum():4d} malignant")

## Passo 8 — comparar no conjunto de validação

**Preveja.** Entre os quatro modelos, qual você espera que tenha a melhor ordenação (ROC AUC e AP)? A diferença vai ser grande?

In [ ]:
from sklearn.metrics import f1_score

rows = []
for name, model in models.items():
    p = model.predict_proba(X_val)[:, 1]
    f = p >= 0.5
    rows.append({'model': name, 'ROC AUC': roc_auc_score(y_val, p), 'AP': average_precision_score(y_val, p),
                 'precision@0.5': precision_score(y_val, f), 'recall@0.5': recall_score(y_val, f),
                 'F1@0.5': f1_score(y_val, f)})
comparison = pd.DataFrame(rows).set_index('model')
print(comparison.round(3).to_string())

**Simulação: comparar no limiar justo.** A tabela acima corta os quatro modelos em 0,5, o que é injusto: cada um põe seus escores numa escala diferente. Refaça a comparação dando a **cada modelo o seu próprio limiar**, o que atinge recall ≥ 0,80 na validação com a melhor precisão. As diferenças aumentam ou diminuem?

In [ ]:
# sua simulação aqui

## Passo 9 — escolher o modelo e o limiar na validação

**Preveja.** Dado o que o passo 8 mostrou, qual modelo deve ser escolhido? E o limiar que atinge recall 0,80 vai ficar perto de 0,5 ou bem abaixo?

In [ ]:
best_name = comparison['AP'].idxmax()
best = models[best_name]
p_val_best = best.predict_proba(X_val)[:, 1]
prec, rec, thr = precision_recall_curve(y_val, p_val_best)
ok = rec[:-1] >= 0.80                            # limiares que atingem a meta de recall
i = np.argmax(np.where(ok, prec[:-1], -1))       # entre eles, a melhor precisão
threshold = thr[i]
print(f"model: {best_name}   threshold {threshold:.4f}")
print(f"validation: recall {rec[i]:.3f}   precision {prec[i]:.3f}   flagged {(p_val_best >= threshold).sum()}")

## Passo 10 — o único teste

**Preveja.** O recall no teste vai ficar acima ou abaixo dos 0,808 da validação? Com 52 casos malignos, quanto vale um caso?

In [ ]:
from sklearn.metrics import confusion_matrix

p_test = best.predict_proba(X_test)[:, 1]
pred_test = (p_test >= threshold).astype(int)
print(confusion_matrix(y_test, pred_test, labels=[1, 0]))
print(f"recall {recall_score(y_test, pred_test):.3f}   precision {precision_score(y_test, pred_test):.3f}"
      f"   flagged {pred_test.sum()} of {len(pred_test)}")
print(f"ROC AUC {roc_auc_score(y_test, p_test):.3f}   AP {average_precision_score(y_test, p_test):.3f}")

## Passo 11 — a escada grosseira de um k-NN

**Preveja.** Com $k = 5$, quantas probabilidades distintas o k-NN pode produzir? Quantos pontos a curva ROC dele vai ter e isso é um problema para a nossa meta de recall 0,80?

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

knn = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5)).fit(X_train, y_train)
p_knn = knn.predict_proba(X_val)[:, 1]
print("distinct probabilities:", np.unique(p_knn))
fpr_k, tpr_k, thr_k = roc_curve(y_val, p_knn)
print(f"ROC points: {len(thr_k)}   ROC AUC {roc_auc_score(y_val, p_knn):.3f}   AP {average_precision_score(y_val, p_knn):.3f}")
for t, f, r in zip(thr_k, fpr_k, tpr_k):
    print(f"threshold {t:.1f}   FPR {f:.3f}   recall {r:.3f}")

fig, ax = plt.subplots(figsize=(5, 4))
RocCurveDisplay.from_predictions(y_val, p_val, ax=ax, name='logistic')
RocCurveDisplay.from_predictions(y_val, p_knn, ax=ax, name='5-NN')
plt.show()

**Para fechar a aula.** Em uma frase cada: (1) o que os pesos de classe mudaram e o que não mudaram? (2) Por que reamostrar antes de dividir inflaciona o resultado de duas maneiras? (3) Por que a regressão logística simples venceu a comparação?

## Sua vez (avançado)

Para casa, com mais tempo:

1. **Um custo em vez de uma meta.** Suponha que deixar passar um câncer custe 20 vezes uma nova convocação desnecessária. No conjunto de validação, calcule o custo $20 \cdot FN + FP$ para cada limiar do modelo escolhido e encontre o mínimo. Compare com a fórmula $t^* = 1/(1+20)$. Por que eles diferem?
2. **Reamostrar antes de dividir.** Aplique `SMOTE` a todo o `X, y`, depois divida 80/20 e ajuste um 5-NN. Compare seu recall e sua precisão com o passo 10. Quais duas coisas deram errado?
3. **Mais vizinhos.** Repita o passo 11 com $k = 15$ e $k = 51$. Quantas probabilidades distintas, quantos pontos ROC e o que acontece com a AUC? Quais pontos de operação perto de recall 0,80 passam a existir?
4. **A sorte da divisão.** Troque `random_state=7` no passo 2 por 0, 1, 2, 3 e 4 e rode de novo os passos 2–10. Quanto se movem o limiar escolhido e o recall de teste? O que isso diz sobre reportar "recall 0,827" com três casas decimais?